# Milestone 3 · E1 — Bảng gán nhãn

**Đọc hướng dẫn rồi chạy từng cell từ 1 đến 6.** Bảng có tìm kiếm, phân trang,
nút chọn reference và lưu ngay lên file tiến độ. Không cần nhập số trong console.

| Chế độ | Công việc | Dữ liệu người dùng tạo |
|---|---|---|
| `manual` | Hiếu / Anh / Kiên đọc entry và chọn reference, độc lập với auto | Nhãn thủ công |
| `review` | Kiểm tra những entry còn `needs_review` của một lần chạy auto cụ thể | Quyết định review riêng |

Mỗi người cần **ít nhất 20 nhãn xác nhận cho mỗi bài trong 5 bài**; `Chưa chắc / Tiếp`
không tạo nhãn và không tính vào quota. Kế hoạch hiện chọn 100 entry/người từ cùng dải
5.000 bài. Dữ liệu và quota nhóm còn phải đối soát khi có các dải khác.

**Giữ nguyên dữ liệu đầu vào và file kế hoạch khi tiếp tục tiến độ cũ.** DOI bổ sung
chỉ phục vụ hiển thị, không sửa pool hay làm mất các nhãn đã lưu bằng giao diện CLI.
Không xem trước dự đoán auto khi gán manual. Không tự coi DOI preprint khác DOI tạp chí
là hai tài liệu khác nhau.

## Cell 1 — Cấu hình người dùng và đường dẫn

Chỉ sửa các biến trong cell này. **Chạy cell sau khi sửa** để cấu hình có hiệu lực.
Các đường dẫn dữ liệu và nơi lưu nhãn được để trống: mỗi người tự điền theo Drive/máy
của mình. Không cần đặt dữ liệu theo tên thư mục hay cấu trúc của người khác.

| Biến | Ý nghĩa / cách điền |
|---|---|
| `DEMO` | `True` để thử bằng dữ liệu giả, không ghi vào tiến độ thật. `False` để làm việc thật. |
| `MODE` | `manual` hoặc `review`. Manual không nạp kết quả dự đoán auto. |
| `MEMBER` | Hiếu=`member1`, Anh=`member2`, Kiên=`member3`; chọn phần việc trong kế hoạch. |
| `REVIEWER` | Mã người thực sự chọn nhãn: `hieu`, `anh`, `kien`. Không dùng chung mã. |
| `PROJECT_ROOT` | Nơi chứa **code**, phải có `src/matching`. Trên Colab code được clone vào `/content/arxiv_m3_e1`. |
| `BIBTEX_PATH` | File BibTeX M2 đã parse. Không phải references M1. |
| `REFERENCES_PATH` | File `references.parquet` trong snapshot export M1 đã chốt. |
| `PLAN_PATH` | File `dataset_plan.json` đã phân công, cần upload lên Drive; GitHub không chứa file này. |
| `DOI_INDEX` | File `doi_index.parquet` đã phục hồi. Nếu chưa có, đặt `None`; giao diện vẫn chạy và hiện trường thiếu. |
| `PROGRESS_ROOT` | Bắt buộc điền một **thư mục** được phép ghi nhãn; không điền tên file Parquet. Có thể dùng Drive riêng, tách khỏi nơi đọc dataset. |
| `AUTO_RUN_DIR` | Chỉ dùng khi `MODE='review'`: đúng thư mục kết quả auto v2 cần review, có `_SUCCESS` và `summary.json`. |

### Dùng chung dataset hay dùng data riêng?
Mỗi người tự điền đường dẫn tới file trên Drive/máy mình. Nếu dùng cùng kế hoạch nhóm,
hai bảng M1/M2 phải cùng nội dung dù nằm ở các đường dẫn khác nhau.
Nếu dùng dải khác hoặc snapshot khác nội dung, phải có kế hoạch tương ứng với dữ liệu đó
và MEMBER phải có phân công trong kế hoạch; không bỏ kiểm tra fingerprint.

`MyDrive` là **Drive của tôi** của tài khoản đã mount. Link thư mục Drive không tự biến
thành đường dẫn `/content/drive/...`. Với thư mục được chia sẻ, chuẩn bị shortcut/bản sao
mà tài khoản của bạn có thể truy cập; dùng **Copy path** trong Files của Colab để điền.

**Cách điền:** mở Files bên trái Colab → tìm file → Copy path → dán chuỗi vào biến.
Ví dụ minh họa: `BIBTEX_PATH = '/content/drive/MyDrive/du_lieu_cua_ban/bibtex.parquet'`.
Điền đường dẫn **đến file** cho BIBTEX_PATH / REFERENCES_PATH / PLAN_PATH / DOI_INDEX;
điền đường dẫn **đến thư mục** cho PROGRESS_ROOT / AUTO_RUN_DIR. Không dán link HTTPS Drive.

Trên máy local có thể điền `r'D:/thu_muc_cua_ban/bibtex.parquet'`; đường dẫn ổ D không
dùng được với runtime Colab. Ví dụ chỉ để hướng dẫn, không phải dữ liệu điền sẵn.

In [ ]:
from pathlib import Path
import sys

try:
    import google.colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

DEMO = False
MODE = 'manual'
MEMBER = ''         # TỰ ĐIỀN: Hiếu: member1 | Anh: member2 | Kiên: member3
REVIEWER = ''       # TỰ ĐIỀN: hieu / anh / kien; đúng người thực sự chọn nhãn
PAGE_SIZE = 10       # Số ứng viên trên một trang; vẫn truy cập được toàn bộ ứng viên

# CODE: nơi phải có src/matching. Khác với thư mục chứa dữ liệu.
PROJECT_ROOT = Path('/content/arxiv_m3_e1') if ON_COLAB else Path.cwd()
if not ON_COLAB and not (PROJECT_ROOT / 'src/matching').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

UPDATE_CODE = True       # Colab: pull --ff-only nhánh M3_E1; không dùng force/reset
INSTALL_PACKAGES = ON_COLAB

# BẮT BUỘC: Copy path đến FILE bibtex.parquet do M2 xuất ra.
BIBTEX_PATH = ''

# BẮT BUỘC: Copy path đến FILE references.parquet của snapshot M1 đã chọn.
REFERENCES_PATH = ''

# BẮT BUỘC: Copy path đến FILE dataset_plan.json phù hợp với hai bảng trên.
PLAN_PATH = ''

# BẮT BUỘC: THƯ MỤC được phép ghi, để lưu manual/memberX.parquet bên trong.
# Ví dụ minh họa: '/content/drive/MyDrive/nhan_cua_ban'
# Nếu tiếp tục nhãn cũ: điền đúng thư mục cha chứa folder manual/ trước đây.
PROGRESS_ROOT = ''

# TÙY CHỌN: Copy path đến FILE doi_index.parquet; để '' hoặc None nếu chưa có.
DOI_INDEX = ''

# CHỈ BẮT BUỘC KHI MODE='review': THƯ MỤC của đúng lần chạy auto cần review.
# Thư mục phải có _SUCCESS, summary.json và auto_labels.parquet. Manual để trống.
AUTO_RUN_DIR = ''

print('Môi trường:', 'Colab' if ON_COLAB else 'Local')
print('Chế độ:', 'Demo' if DEMO else MODE)
print('Người dùng:', MEMBER, '/', REVIEWER)
print('Code:', PROJECT_ROOT)
print('BibTeX:', BIBTEX_PATH or '[chưa điền]')
print('References:', REFERENCES_PATH or '[chưa điền]')
print('Nơi lưu nhãn:', PROGRESS_ROOT)

## Cell 2 — Mount Drive, chuẩn bị code, cài thư viện và import

Trên Colab, đăng nhập tài khoản có quyền đọc dataset và ghi nhãn khi hộp mount xuất hiện.
Cell clone/pull nhánh `M3_E1`, cài requirements cho bảng (`ipywidgets`) và thêm `src` vào
đường dẫn import. **Mở notebook từ GitHub chưa tự tải source code vào runtime.**

Trên local không mount hoặc pull; nếu chưa cài thư viện, bật `INSTALL_PACKAGES=True` ở Cell 1
hoặc chạy `python -m pip install -r notebooks/requirements-labeling.txt` bằng đúng kernel.

Khi chạy lại trong cùng phiên, bảng đang mở được đóng trước khi reload module để tránh nút
cũ ghi nhãn ngoài ý muốn. Nhãn đã lưu vẫn còn. Sau reset runtime, chạy lại từ Cell 1.
Không chạy đồng thời hai phiên cho cùng một file tiến độ, đặc biệt trên Drive.

In [ ]:
import importlib
import subprocess

if 'labeling_ui' in globals():
    labeling_ui.close()

if ON_COLAB:
    from google.colab import drive, output as colab_output
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    if not PROJECT_ROOT.exists():
        subprocess.run(['git', 'clone', '--branch', 'M3_E1', '--single-branch',
            'https://github.com/NewB-S2deeC0de/arXiv.git', str(PROJECT_ROOT)], check=True)
    elif UPDATE_CODE:
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only',
                        'origin', 'M3_E1'], check=True)

assert (PROJECT_ROOT / 'src/matching/labeling_table.py').is_file(), (
    f'Chưa có code bảng tại {PROJECT_ROOT}. Kiểm tra clone/pull và PROJECT_ROOT ở Cell 1.'
)

if INSTALL_PACKAGES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
        str(PROJECT_ROOT / 'notebooks/requirements-labeling.txt')], check=True)

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
importlib.invalidate_caches()
import matching.labeling as labeling_module
import matching.auto_matcher as matcher_module
import matching.labeling_table as table_module
for module in (labeling_module, matcher_module, table_module):
    importlib.reload(module)
assert Path(table_module.__file__).resolve().parent == (PROJECT_ROOT / 'src/matching').resolve(), (
    'Python đang nạp matching từ repo khác. Restart kernel rồi chạy lại từ Cell 1.'
)

import json
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
from IPython.display import display, HTML
from matching.candidate_pool import build_candidate_pool
from matching.labeling import pool_fingerprint
from matching.labeling_table import show_labeling_table

if ON_COLAB:
    colab_output.enable_custom_widget_manager()

print('Đã nạp bảng gán nhãn:', table_module.__file__)
print('Policy matcher đang có trong source:', matcher_module.POLICY['name'])

## Cell 3 — Kiểm tra file, snapshot và phân công

Cell đọc BibTeX / references, xác nhận fingerprint khớp kế hoạch và chọn đúng 100 entry
manual của MEMBER. **Chưa tạo nhãn ở bước này.** Nhãn cũ chỉ được kiểm tra/nạp ở Cell 4.

Manual chỉ đọc kế hoạch, không đọc auto labels hay confidence. Review yêu cầu marker hoàn tất,
summary khớp snapshot và lấy những entry `needs_review` trực tiếp từ `auto_labels.parquet`
của `AUTO_RUN_DIR`, tránh dùng nhầm review queue cũ.

DOI index được giữ ngoài pool: DOI chỉ bổ sung thông tin hiển thị. Nếu chỉ có một phần DOI,
những trường còn thiếu vẫn hiển thị `[thiếu dữ liệu]`. Không đổi fingerprint để ép dùng lại nhãn.

**Tiến độ manual:** `PROGRESS_ROOT/manual/member1.parquet` (tương ứng member2/member3).
**Tiến độ review:** `PROGRESS_ROOT/review/<tên-lần-auto>/<REVIEWER>.parquet`.
Mỗi người cần quyền ghi vào PROGRESS_ROOT. Không nhập dấu `/`, `\` trong mã người dùng.

In [ ]:
import re

if 'labeling_ui' in globals():
    labeling_ui.close()

if DEMO:
    import tempfile
    demo_directory = tempfile.TemporaryDirectory(prefix='m3-table-demo-')
    progress = Path(demo_directory.name) / 'demo.parquet'
    reviewer, publications, entry_keys = 'demo', None, None
    pool = build_candidate_pool(pa.Table.from_pylist([
        dict(arxiv_id='demo-paper', citation_key='method', title='Methods for reference matching',
             authors=['Jane Doe'], year=2024, doi='10.1234/example'),
        dict(arxiv_id='demo-paper', citation_key='unknown', title='An unavailable reference',
             authors=['Alex Smith']),
        dict(arxiv_id='demo-empty', citation_key='empty', title='Entry with no candidates')]),
        pa.Table.from_pylist([
            dict(arxiv_id='demo-paper', ref_arxiv_id='2401.00001',
                 ref_title='Methods for reference matching', ref_authors=['Jane Doe'],
                 ref_submission_date='2024-01-02'),
            dict(arxiv_id='demo-paper', ref_arxiv_id='2402.00002',
                 ref_title='Another topic', ref_authors=['Other Person'])]))
    display_dois = pa.table({'arxiv_id': ['2401.00001'], 'doi': ['10.1234/example']})
    plan = None
else:
    if MODE not in ('manual', 'review'):
        raise ValueError('MODE phải là manual hoặc review')
    for name, value in [('MEMBER', MEMBER), ('REVIEWER', REVIEWER)]:
        if not re.fullmatch(r'[A-Za-z0-9_-]+', value):
            raise ValueError(f'{name}: dùng mã không rỗng, chỉ gồm chữ không dấu, số, _ hoặc -')
    path_values = {'BIBTEX_PATH': BIBTEX_PATH, 'REFERENCES_PATH': REFERENCES_PATH,
                   'PLAN_PATH': PLAN_PATH, 'PROGRESS_ROOT': PROGRESS_ROOT}
    if MODE == 'review':
        path_values['AUTO_RUN_DIR'] = AUTO_RUN_DIR
    blank = [name for name, value in path_values.items()
             if value is None or not str(value).strip()]
    if blank:
        raise ValueError('Chưa điền đường dẫn ở Cell 1: ' + ', '.join(blank))
    BIBTEX_PATH = Path(BIBTEX_PATH).expanduser()
    REFERENCES_PATH = Path(REFERENCES_PATH).expanduser()
    PLAN_PATH = Path(PLAN_PATH).expanduser()
    PROGRESS_ROOT = Path(PROGRESS_ROOT).expanduser()
    DOI_INDEX = Path(DOI_INDEX).expanduser() if DOI_INDEX is not None and str(DOI_INDEX).strip() else None
    if MODE == 'review':
        AUTO_RUN_DIR = Path(AUTO_RUN_DIR).expanduser()
    if PROGRESS_ROOT.exists() and not PROGRESS_ROOT.is_dir():
        raise NotADirectoryError('PROGRESS_ROOT phải là thư mục lưu nhãn, không phải tên file')
    required = {'BibTeX M2': BIBTEX_PATH, 'References M1': REFERENCES_PATH,
                'Kế hoạch': PLAN_PATH}
    if DOI_INDEX is not None:
        required['DOI bổ sung'] = DOI_INDEX
    missing = []
    for name, path in required.items():
        exists = Path(path).is_file()
        print('✓' if exists else '✗', name, ':', path)
        if not exists:
            missing.append(str(path))
    if missing:
        raise FileNotFoundError('Sửa đường dẫn ở Cell 1 bằng Copy path, rồi chạy lại từ Cell 1:\n'
                                + '\n'.join(missing))
    plan = json.loads(PLAN_PATH.read_text(encoding='utf-8'))
    print('Đang dựng candidate pool...', flush=True)
    pool = build_candidate_pool(BIBTEX_PATH, REFERENCES_PATH)
    if pool_fingerprint(pool) != plan['candidate_pool_sha256']:
        raise ValueError('Snapshot khác kế hoạch. Kiểm tra đúng input; không sửa/bỏ fingerprint.')
    reviewer = REVIEWER
    display_dois = DOI_INDEX
    if MODE == 'manual':
        assigned = [a for a in plan['manual_assignments'] if a['owner'] == MEMBER]
        if not assigned:
            raise ValueError(f'MEMBER={MEMBER} chưa có trong kế hoạch phân công')
        publications = [a['arxiv_id'] for a in assigned]
        entry_keys = [(a['arxiv_id'], k) for a in assigned for k in a['citation_keys']]
        progress = PROGRESS_ROOT / 'manual' / f'{MEMBER}.parquet'
    else:
        for filename in ('_SUCCESS', 'summary.json', 'auto_labels.parquet'):
            if not (AUTO_RUN_DIR / filename).is_file():
                raise FileNotFoundError(f'Auto chưa hoàn tất hoặc sai AUTO_RUN_DIR: {filename}')
        auto_report = json.loads((AUTO_RUN_DIR / 'summary.json').read_text(encoding='utf-8'))
        if auto_report['candidate_pool_sha256'] != plan['candidate_pool_sha256']:
            raise ValueError('Auto run thuộc snapshot khác kế hoạch')
        auto_labels = pq.read_table(AUTO_RUN_DIR / 'auto_labels.parquet')
        if not auto_labels.schema.equals(matcher_module.AUTO_LABEL_SCHEMA):
            raise ValueError('Schema auto labels không đúng')
        queue = auto_labels.filter(pc.equal(auto_labels['label_status'], 'needs_review')).to_pylist()
        entry_keys = [(r['arxiv_id'], r['citation_key']) for r in queue]
        publications = sorted({r['arxiv_id'] for r in queue})
        progress = PROGRESS_ROOT / 'review' / AUTO_RUN_DIR.name / f'{REVIEWER}.parquet'
        print('Auto được chọn:', AUTO_RUN_DIR)
        print('Policy của auto run:', auto_report['policy']['name'])
        if not entry_keys:
            raise ValueError('Auto run này không còn entry needs_review để mở bảng')

print('Người gán nhãn:', reviewer)
print('File tiến độ:', progress)
print('Số entry được chọn:', len(entry_keys) if entry_keys is not None else pool.entries.num_rows)
print('Tiến độ đã tồn tại:', progress.is_file())
for row in pool.summary.to_pylist():
    if publications is None or row['arxiv_id'] in publications:
        print(row)

## Cell 4 — Mở bảng và gán nhãn

Chạy cell rồi thao tác trực tiếp trên bảng. Cell có thể kết thúc hiển thị dấu ✓ nhưng các
nút vẫn hoạt động nhờ kết nối widget; không cần chạy lại sau mỗi nhãn.

1. Đọc **ENTRY CẦN GÁN NHÃN** ở phía trên: title, authors, year, DOI và nguồn.
2. Gõ từ đặc trưng trong title/tác giả/DOI/arXiv ID vào ô tìm kiếm rồi nhấn Enter hoặc rời ô.
   Các từ khóa phải cùng xuất hiện trong metadata ứng viên. Tìm không ra chưa có nghĩa là no-match.
3. Đối chiếu dòng ứng viên; mở arXiv nếu cần. Nút `Chọn` **lưu ngay và chuyển sang entry chưa gán**.
4. Nếu không chắc, dùng `Chưa chắc / Tiếp`. Không tạo nhãn mới, giữ nhãn cũ nếu đã có.

| Điều khiển | Tác dụng |
|---|---|
| `Xem tất cả` | Xóa bộ lọc, giữ toàn bộ candidate scope ban đầu. |
| `← Trang` / `Trang →` | Xem các trang ứng viên; không thay đổi entry/nhãn. |
| `Không có match` | Xác nhận nhãn âm, lưu và chuyển tiếp. Bị khóa khi đang lọc. Cần xem đủ các trang trước khi kết luận. |
| `← Entry` | Quay lại kiểm tra/sửa entry trước. |
| `Entry chưa gán` | Tìm entry còn pending, gồm các entry đã bỏ qua. |
| `Xóa nhãn` | Xóa quyết định của entry hiện tại, đưa về chưa gán. |
| `Nạp lại tiến độ` | Đọc file trên Drive sau khi có thay đổi bên ngoài hoặc báo xung đột. |
| Hai danh sách chọn | Đi thẳng tới publication hoặc entry trong phân công. ✓ là đã lưu, ○ là chưa gán. |

DOI bổ sung hiển thị nguồn `Bảng DOI bổ sung` và mọi giá trị
khác nhau nếu có. Đây là metadata để bạn đối chiếu, **không phải gợi ý chọn nhãn**.
Tên hơn ba tác giả có thể mở rộng để xem đầy đủ. Khi thiếu metadata, đọc source hoặc
để pending; không đoán no-match. Nếu 20 entry đã chọn có ca chưa xác minh được, cần giải
quyết hoặc bổ sung kế hoạch có đối soát trước khi đạt quota; không đổi tùy tiện keys trong JSON.

Nếu không thấy bảng, kiểm tra widget manager, kết nối runtime và chạy lại Cell 2 → 3 → 4.
Không chạy `run_labeling_session` dạng console cùng lúc với bảng trên cùng file.

In [ ]:
if 'pool' not in globals() or 'progress' not in globals():
    raise RuntimeError('Chạy Cell 1 → 2 → 3 trước khi mở bảng')
if 'labeling_ui' in globals():
    labeling_ui.close()

labeling_ui = show_labeling_table(
    pool, progress,
    reviewer=reviewer,
    publications=publications,
    entry_keys=entry_keys,
    papers=display_dois,
    page_size=PAGE_SIZE,
)

## Cell 5 — Kiểm tra tiến độ thật đã lưu

Chạy bất cứ lúc nào sau Cell 4. Cell nạp file tiến độ và đếm theo từng publication;
không dùng số entry đã xem làm số nhãn. Việc mở Cell 3 chưa tạo nhãn. File có thể chứa
nhãn cũ ngoài phạm vi UI hiện tại; các nhãn đó vẫn được giữ nguyên.

Manual đạt phần việc hiện tại khi **mỗi bài có ít nhất 20 nhãn xác nhận**. Với kế hoạch
100 entry thì cần đủ 100; skip chưa được tính. Review không có quota 20/bài, nhưng mọi
entry cần review phải được xử lý trước khi publication auto được nghiệm thu hoàn tất.

In [ ]:
if 'labeling_ui' not in globals():
    raise RuntimeError('Chạy Cell 4 trước')
session = labeling_ui.session
session.refresh()
rows = session.progress_rows()
display(HTML('<table><tr><th>Bài báo</th><th>Đã xác nhận</th><th>Được chọn</th><th>Chưa gán</th></tr>'
    + ''.join(f"<tr><td>{r['arxiv_id']}</td><td>{r['confirmed']}</td>"
              f"<td>{r['selected_entries']}</td><td>{r['pending']}</td></tr>" for r in rows)
    + '</table>'))
print('Tổng nhãn trong phân công:', sum(r['confirmed'] for r in rows), '/', len(session.keys))
print('File lưu:', session.progress)
if not DEMO and MODE == 'manual':
    print('Bài đạt ít nhất 20 nhãn:', sum(r['confirmed'] >= 20 for r in rows), '/', len(rows))
print('Tiến độ CLI cũ được giữ nguyên; DOI hiển thị không thay fingerprint.')

## Cell 6 — Kết thúc phiên và bàn giao

Mỗi lần chọn đã được ghi ngay. Chạy cell này để đóng bảng và đọc lại metadata file
đã lưu. Không tải toàn bộ file lên RAM hoặc tự gửi cho thành viên khác.

Gửi **file Parquet tiến độ** cho người tổng hợp; không chỉ gửi ảnh notebook. Nếu dùng
Drive chung, báo đúng đường dẫn file. Không rename/ghi đè file của người khác.
Nhóm phải kiểm tra đủ 15 bài manual, ≥300 nhãn, các review auto đã xử lý và corpus/quota
cuối trước khi dùng `matching.groundtruth` để chia tập seed 42. Notebook không tự xuất
ground truth hay tự đổi các nhãn đang mâu thuẫn.

Để tiếp tục lần sau: chạy Cell 1 → 2 → 3 → 4 bằng **cùng đường dẫn tiến độ và input**.
Ba nhãn đã lưu từ CLI trước đây sẽ được giữ; không cần gán lại từ đầu.

In [ ]:
if 'labeling_ui' not in globals():
    raise RuntimeError('Chưa mở bảng ở Cell 4')
labeling_ui.session.refresh()
final_progress = labeling_ui.session.progress
labeling_ui.close()
if final_progress.is_file():
    with pq.ParquetFile(final_progress) as saved:
        print('Đã đóng bảng. File tiến độ:', final_progress)
        print('Tổng nhãn trong file:', saved.metadata.num_rows)
else:
    print('Đã đóng bảng. Chưa có nhãn xác nhận nên chưa tạo file tiến độ.')

## Tự khắc phục lỗi thường gặp

| Lỗi | Cách xử lý |
|---|---|
| `No module named matching` / hàm chưa định nghĩa | Chạy lại Cell 2; không chỉ mở file labeling.py trong Files. |
| Không tìm thấy src/matching | Kiểm tra PROJECT_ROOT, clone/pull đúng M3_E1; không dùng thư mục dataset làm đường dẫn code. |
| Chưa điền đường dẫn | Điền các biến được nêu tên ở Cell 1, chạy lại Cell 1 rồi Cell 3. Để trống DOI_INDEX được phép; AUTO_RUN_DIR chỉ bắt buộc với review. |
| Không tìm thấy dataset_plan.json | Upload đúng file vào PLAN_PATH; tên file là dataset_plan.json. |
| FileNotFoundError Parquet/DOI | Copy path trong Files của Colab và sửa Cell 1; nếu chưa có DOI, đặt DOI_INDEX=None. |
| Fingerprint khác | Input khác phiên bản kế hoạch/nhãn; xác minh lại snapshot. Không bỏ kiểm tra hay sửa hash. |
| Git pull không fast-forward hoặc có local changes | Giữ bản sửa local để kiểm tra; không reset/force. Có thể đặt UPDATE_CODE=False nếu đã có đúng source. |
| Lock hoặc entry được sửa ở phiên khác | Dừng phiên trùng. Bấm Nạp lại tiến độ. Chỉ xóa lock cũ sau khi chắc chắn writer cũ đã dừng. |
| Không thấy nút / nút không phản hồi | Kiểm tra kết nối Colab và widget manager ở Cell 2; mở lại Cell 4. GitHub preview không chạy widget. |
| Drive không ghi được | Kiểm tra quyền và dung lượng; đổi PROGRESS_ROOT sang Drive riêng trước khi bắt đầu. Nếu đã có nhãn, sao lưu/copy tiến độ có chủ đích. |
| Kernel/RAM reset | Mount lại, chạy từ Cell 1; nhãn đã lưu trên Drive còn nguyên. Không rerun crawler/parser/auto chỉ để tiếp tục manual. |

Không tự động đề xuất nhãn, xếp hạng theo matcher hoặc coi kết quả lọc rỗng là no-match.
Giao diện tìm kiếm chỉ giúp bạn đọc nhanh hơn; quyết định vẫn do người gán nhãn xác nhận.